<h1>Extracting Data from Flight Labs API</h1>

In [1]:
import os
from dotenv import load_dotenv
from utils import extract, transform, load_to_csv, load_to_postgres
from sqlalchemy import create_engine

load_dotenv()
access_key = os.getenv('ACCESS_KEY')

# Define API endpoint
url = 'https://www.goflightlabs.com/flights'

# Extracting data from API endpoint
flight_data_raw = extract(url, access_key)
print(flight_data_raw.shape)

Returned status code: 200
Extraction complete
(100, 22)


<H1>Exploring the Raw Data </h1>

In [3]:
# EDA
display(flight_data_raw.head(), flight_data_raw.info(), flight_data_raw.describe())

print('\nNumber of null values for every column feature\n')
flight_data_raw.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   hex            100 non-null    object 
 1   reg_number     100 non-null    object 
 2   flag           100 non-null    object 
 3   lat            100 non-null    float64
 4   lng            100 non-null    float64
 5   alt            100 non-null    int64  
 6   dir            100 non-null    float64
 7   speed          100 non-null    int64  
 8   v_speed        100 non-null    int64  
 9   flight_number  100 non-null    object 
 10  flight_icao    100 non-null    object 
 11  flight_iata    100 non-null    object 
 12  dep_icao       100 non-null    object 
 13  dep_iata       100 non-null    object 
 14  arr_icao       100 non-null    object 
 15  arr_iata       100 non-null    object 
 16  airline_icao   100 non-null    object 
 17  airline_iata   100 non-null    object 
 18  aircraft_ic

,hex,reg_number,flag,lat,lng,alt,dir,speed,v_speed,flight_number,...,dep_icao,dep_iata,arr_icao,arr_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type
0,789280,B-KKD,HK,25.497075,122.466694,11343,50.9,980,0,686,...,VHHH,HKG,RJBB,KIX,HKE,UO,A21N,1764726864,en-route,adsb
1,881421,HS-EAA,TH,17.882290,99.437927,7914,163.3,783,0,3438,...,VTCC,CNX,VTBD,DMK,AIQ,FD,A21N,1764726864,en-route,adsb
2,7C47A6,VH-OFS,AU,-39.992847,145.880419,10901,161.6,842,0,705,...,YMML,MEL,YMHB,HBA,JST,JQ,A21N,1764726864,en-route,adsb
3,78192A,B-20EK,CN,21.836641,112.494869,11076,248.1,770,0,853,...,ZSAM,XMN,VTBS,BKK,CXA,MF,B38M,1764726864,en-route,adsb
4,E80620,CC-DIC,CL,-31.311245,-64.208318,10688,183.0,45,0,3108,...,SABE,AEP,SACO,COR,JES,WJ,A21N,1764726864,landed,adsb


None

,lat,lng,alt,dir,speed,v_speed,updated
count,100.000000,100.000000,100.000000,100.000000,100.00000,100.0,1.000000e+02
mean,17.192191,10.739313,9368.020000,197.422000,771.34000,0.0,1.764727e+09
std,23.490918,101.294855,3284.192735,96.117399,210.20054,0.0,0.000000e+00
min,-39.992847,-127.406431,10.000000,3.500000,24.00000,0.0,1.764727e+09
25%,10.135173,-94.607439,8792.000000,118.600000,727.25000,0.0,1.764727e+09
50%,21.680843,52.462190,10688.000000,185.000000,830.00000,0.0,1.764727e+09
75%,34.510818,109.337031,11415.000000,283.000000,886.25000,0.0,1.764727e+09
max,54.561174,151.662924,13141.000000,357.900000,1172.00000,0.0,1.764727e+09



Number of null values for every column feature



hex              0
reg_number       0
flag             0
lat              0
lng              0
alt              0
dir              0
speed            0
v_speed          0
flight_number    0
flight_icao      0
flight_iata      0
dep_icao         0
dep_iata         0
arr_icao         0
arr_iata         0
airline_icao     0
airline_iata     0
aircraft_icao    0
updated          0
status           0
type             0
dtype: int64

<h1>Data Cleaning</h1>

<li>Replacing missing values in "squawk" column with "unknown" if the column is pulled during extraction</li>
<li>Replacing missing values in 'alt', 'speed' and v_speed to 0</li>
<li>Renaming columns</li>
<li>Converting "updated" values to datetime</li>


In [4]:
flight_data_clean = transform(flight_data_raw)

display(flight_data_clean.head())

print('\nNumber of null values for every column feature\n')
flight_data_clean.isnull().sum()


transform complete


,hex,reg_number,flag,latitude,longitude,altitude_ft,dir,speed_mph,v_speed_mph,flight_number,...,departure_icao,departure_iata,arrival_icao,arrival_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type
0,789280,B-KKD,HK,25.497075,122.466694,37214.56812,50.9,608.943580,0.0,686,...,VHHH,HKG,RJBB,KIX,HKE,UO,A21N,2025-12-02 20:54:24,en-route,adsb
1,881421,HS-EAA,TH,17.882290,99.437927,25964.56776,163.3,486.533493,0.0,3438,...,VTCC,CNX,VTBD,DMK,AIQ,FD,A21N,2025-12-02 20:54:24,en-route,adsb
2,7C47A6,VH-OFS,AU,-39.992847,145.880419,35764.43684,161.6,523.194382,0.0,705,...,YMML,MEL,YMHB,HBA,JST,JQ,A21N,2025-12-02 20:54:24,en-route,adsb
3,78192A,B-20EK,CN,21.836641,112.494869,36338.58384,248.1,478.455670,0.0,853,...,ZSAM,XMN,VTBS,BKK,CXA,MF,B38M,2025-12-02 20:54:24,en-route,adsb
4,E80620,CC-DIC,CL,-31.311245,-64.208318,35065.61792,183.0,27.961695,0.0,3108,...,SABE,AEP,SACO,COR,JES,WJ,A21N,2025-12-02 20:54:24,landed,adsb



Number of null values for every column feature



hex               0
reg_number        0
flag              0
latitude          0
longitude         0
altitude_ft       0
dir               0
speed_mph         0
v_speed_mph       0
flight_number     0
flight_icao       0
flight_iata       0
departure_icao    0
departure_iata    0
arrival_icao      0
arrival_iata      0
airline_icao      0
airline_iata      0
aircraft_icao     0
updated           0
status            0
type              0
dtype: int64

<h1>Loading Cleaned Data to CSVs and Postgres </h1>

In [5]:
# Connecting to local flight data database
dbname=os.getenv('DB_NAME')
user=os.getenv('DB_USER')
password=os.getenv('DB_PASSWORD')
host=os.getenv('DB_HOST')
port=os.getenv('DB_PORT')

conn = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}')

# Loading raw and clean data
load_to_csv(flight_data_raw, flight_data_clean)
load_to_postgres(flight_data_raw, flight_data_clean, conn)

load complete
